In [53]:
import json
import pandas as pd
import urllib.request
import uuid

import Constants

In [54]:
api_key = Constants.NPS_API_KEY

In [55]:
BASE_URL = "developer.nps.gov/v1"
HEADERS = {"X-Api-Key": api_key}

In [ ]:
## GET BASIC CA NATIONAL PARKS INFORMATION

# Configure API request
parks_endpoint = "https://developer.nps.gov/api/v1/parks?stateCode=ca"

req = urllib.request.Request(parks_endpoint, headers=HEADERS)

with urllib.request.urlopen(req) as response:
    if response.status == 200:
        print(f"STATUS CODE: {response.status}: Request successful!")
        park_ids = []
        park_names = []
        latitudes = []
        longitudes = []
        phone_numbers = []
        emails = []
        
        html = response.read().decode('utf-8')
        json_data = json.loads(html)
        for data in json_data["data"]:
            try:
                park_names.append(data["fullName"])
                # park_ids.append(int(uuid.uuid4()))
                park_ids.append(data["parkCode"])
                latitudes.append(data["latitude"])
                longitudes.append(data["longitude"])
                phone_numbers.append(data["contacts"]["phoneNumbers"][0]["phoneNumber"])
                emails.append(data["contacts"]["emailAddresses"][0]["emailAddress"])
            except:
                print("Parsing issue... continue...")
        # response_df = pd.DataFrame.from_dict(json.loads(html))
        print("Data parsed!")
        print("Creating data file!")
        pd.DataFrame({
            "parkID": park_ids,
            "parkName": park_names,
            "latitude": latitudes,
            "longitude": longitudes,
            "phoneNumber": phone_numbers,
            "email": emails
        }).dropna().to_csv("data/national_parks_coded.csv", index=False)
        print("Data file created! Check /data directory.")

    else:
        print("Something went wrong!")


STATUS CODE: 200: Request successful!
Data parsed!
Creating data file!
Data file created! Check /data directory.


In [89]:
# Get fees and passes information

fees_endpoint = "https://developer.nps.gov/api/v1/feespasses?stateCode=ca"

req = urllib.request.Request(fees_endpoint, headers=HEADERS)

with urllib.request.urlopen(req) as response:
    if response.status == 200:
        print(f"STATUS CODE: {response.status}: Request successful!")

        park_ids = []
        fee_types = []
        entities = []
        costs = []
        descriptions = []

        html = response.read().decode('utf-8')
        json_data = json.loads(html)
        for data in json_data["data"]:
            park_code = data["parkCode"]
            for fee in data["fees"]:
                fee_type, entity = str(fee["entranceFeeType"]).split("-", maxsplit=1)
                cost = float(fee["cost"])
                description = str(fee["description"])
                
                park_ids.append(park_code)
                fee_types.append(fee_type)
                entities.append(entity)
                costs.append(cost)
                descriptions.append(description)
        print("Data parsed!")
        print("Creating fees data file...")
        pd.DataFrame({
            "parkID": park_ids,
            "type": fee_types,
            "entity": entities,
            "cost": costs,
            "description": descriptions
        }).dropna().to_csv("data/fees.csv", index=False)
        print("Data file created! Check /data directory.")

    else:
        print("Something went wrong!")


STATUS CODE: 200: Request successful!
Data parsed!
Creating fees data file...
Data file created! Check /data directory.


In [91]:
for data in sample_data:
    pc = data["parkCode"]
    if data["fees"]:
        print(pc)

cabr
deva
jotr
labe
lavo
pinn
seki
whis
yose


In [79]:
sample_data[21]

{'parkCode': 'pinn',
 'isFeeFreePark': False,
 'isInteragencyPassAccepted': True,
 'cashless': 'Yes',
 'feesAtWorkUrl': 'https://www.nps.gov/pinn/learn/management/yourdollarsatwork.htm',
 'entranceFeeDescription': 'A park entrance pass is required year-round at Pinnacles National Park. Visitors entering the park have multiple pass and entrance fee options available depending on how they arrive to the park. All vehicles must display a pass clearly visible through the windshield. Display federal lands passes, such as annual and military passes, on your dashboard with the signature and expiration date facing up. Passes are non-transferable and passholder must be present.',
 'entrancePassDescription': 'The Pinnacles Annual Pass is valid only at Pinnacles and may be purchased online or in person. You do not need an additional entrance pass if you already have a federal lands pass. It is valid for 12 months from purchase month. This pass admits the pass holder and passengers in a non-commerc

In [91]:
response_data[0]['addresses']

[{'postalCode': '94133',
  'city': 'San Francisco Bay',
  'stateCode': 'CA',
  'countryCode': 'US',
  'provinceTerritoryCode': '',
  'line1': 'Alcatraz Island',
  'type': 'Physical',
  'line3': '',
  'line2': ''},
 {'postalCode': '94123',
  'city': 'San Francisco',
  'stateCode': 'CA',
  'countryCode': 'US',
  'provinceTerritoryCode': '',
  'line1': 'Alcatraz Island',
  'type': 'Mailing',
  'line3': '201 Fort Mason',
  'line2': 'Golden Gate National Recreation Area'}]

In [88]:
print(f"Activities in {response_data[0]['fullName']}\n-----------")
for r in response_data[0]["activities"]:
    print(r["name"])

Activities in Alcatraz Island
-----------
Food
Wildlife Watching
Birdwatching
Museum Exhibits
Shopping
Bookstore and Park Store


In [89]:
print(f"Exception Hours in {response_data[0]['fullName']}\n-----------")
for h in response_data[0]["operatingHours"][0]["exceptions"]:
    print(h["name"], h["startDate"], h["endDate"])
    

Exception Hours in Alcatraz Island
-----------
Thanksgiving 2025-11-27 2025-11-27
Christmas 2025-12-25 2025-12-25
New Year's 2026-01-01 2026-01-01


In [90]:
print(f"Topics in {response_data[0]['fullName']}\n-----------")
for topic in response_data[0]["topics"]:
    print(topic["name"])

Topics in Alcatraz Island
-----------
Colonization and Settlement
Forts
Incarceration
Jails and Prisons
Maritime
Maritime - Military
Lighthouses
Medicine
Hospital
Military
US Army
Native American Heritage
Social Movements
Wars and Conflicts
Tribal Conflicts
Animals
Birds
